# Dartboard Retrieval：兼顾相关性与多样性的 RAG 检索

在知识库“很稠密”（大量相似/重复内容）时，普通 top-k 向量检索很容易返回**很多内容高度重复**的片段。

Dartboard Retrieval（来自论文 *Better RAG using Relevant Information Gain* 的思想）做的事情很直接：

- **先取一批候选**（比如 top-N）
- 在候选中做一个贪心选择：每次挑一个，让最终的 top-k **既相关**，又和已选内容 **尽量不重复**

下面我们会先复现“稠密数据集下 top-k 的重复问题”，然后实现 Dartboard 的贪心选择算法。

## 1) 导入依赖

我们继续用同一份示例 PDF：`Understanding_Climate_Change.pdf`。

- `pypdf`：抽取 PDF 文本
- `FAISS + OpenAIEmbeddings`：向量库
- `numpy`：实现距离矩阵与贪心选择

In [1]:
import os
import sys
from pathlib import Path
from typing import List, Tuple

import numpy as np
import requests
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

/tmp/ipykernel_78180/950182982.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 2) 下载并抽取 PDF 文本

我们把整篇 PDF 的文本抽出来，后面切成 chunks 建向量库。

In [2]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
PDF_PATH = DATA_DIR / "Understanding_Climate_Change.pdf"

if not PDF_PATH.exists():
    resp = requests.get(
        PDF_URL,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    resp.raise_for_status()
    PDF_PATH.write_bytes(resp.content)

reader = PdfReader(str(PDF_PATH))
pages_text = [(p.extract_text() or "") for p in reader.pages]
content = "\n\n".join([t for t in pages_text if t.strip()])

assert len(content.strip()) > 0, "PDF 提取到的文本为空（pypdf.extract_text 可能失败）"
print("Pages:", len(reader.pages))
print("Chars:", len(content))

Pages: 33
Chars: 72592


## 3) 构建一个“稠密”向量库：故意制造重复

为了复现论文里的典型痛点，我们会把同一批 chunks **重复多次**，模拟“知识库很稠密 / 内容高度重叠”的情况。

接下来你会看到：普通 top-k 检索可能返回多个几乎一模一样的 chunks。

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

raw_doc = Document(page_content=content, metadata={"source": str(PDF_PATH)})
base_chunks = splitter.split_documents([raw_doc])

# 模拟“稠密数据集”：把 chunks 复制多份
REPEAT = 5
chunks = base_chunks * REPEAT

# 给每个 chunk 一个稳定 ID（即使内容重复也要可区分）
for i, c in enumerate(chunks):
    c.metadata.update({"chunk_id": i, "repeat": i // len(base_chunks)})

print("Base chunks:", len(base_chunks))
print("Dense chunks:", len(chunks))

Base chunks: 97
Dense chunks: 485


In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 一次性 embed，避免重复调用 embedding
texts = [d.page_content for d in chunks]
vectors = embeddings.embed_documents(texts)
X = np.array(vectors, dtype=np.float32)

# 构建 FAISS（直接用已算好的向量）
vectorstore = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, vectors)),
    embedding=embeddings,
    metadatas=[d.metadata for d in chunks],
)

print("FAISS ntotal:", int(vectorstore.index.ntotal))

FAISS ntotal: 485


## 4) 先看普通 top-k：稠密知识库下的“重复召回”

我们直接用 FAISS 的索引做 top-k 检索，然后把取回的内容打印出来。

In [6]:
def idx_to_doc(idx: int) -> Document:
    docstore_id = vectorstore.index_to_docstore_id[idx]
    return vectorstore.docstore.search(docstore_id)


def show_texts(texts: List[str]) -> None:
    for i, t in enumerate(texts, start=1):
        print(f"Context {i}:")
        print(t)
        print()


def get_context_topk(query: str, k: int = 5) -> List[str]:
    q_vec = embeddings.embed_query(query)
    _, indices = vectorstore.index.search(np.array([q_vec], dtype=np.float32), k=k)
    return [idx_to_doc(int(i)).page_content for i in indices[0]]


test_query = "What is the main cause of climate change?"
texts_topk = get_context_topk(test_query, k=3)
show_texts(texts_topk)

Context 1:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal

Context 2:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, suc

## 5) Dartboard：把“距离”变成可加的分数

Dartboard 的输入只需要两类距离：

- **query-doc 距离**：查询与每个候选文档的距离
- **doc-doc 距离**：候选文档之间两两的距离

然后用一个贪心过程选出 top-k，让它们 **既接近 query（相关）**，又 **彼此离得更远（多样）**。

下面我们用一个简单的 log-normal 形式，把距离转换成“对数分数”（距离越近，分数越大）。

In [7]:
def lognorm(dist: np.ndarray, sigma: float) -> np.ndarray:
    """log N(0, sigma^2) 在 dist 处的对数值（dist 越小越大）。"""
    if sigma < 1e-9:
        return -np.inf * dist
    return -np.log(sigma) - 0.5 * np.log(2 * np.pi) - dist**2 / (2 * sigma**2)


def logsumexp(a: np.ndarray, axis: int = -1) -> np.ndarray:
    """numpy 版 logsumexp，避免额外依赖 scipy。"""
    a = np.asarray(a, dtype=np.float64)
    m = np.max(a, axis=axis, keepdims=True)
    # 若全是 -inf，m 也是 -inf，此时 exp(a-m) 会是 nan，需要单独处理
    stable = np.where(np.isfinite(m), a - m, -np.inf)
    s = np.sum(np.exp(stable), axis=axis)
    out = np.squeeze(m, axis=axis) + np.log(s)
    out = np.where(np.isfinite(out), out, -np.inf)
    return out


def normalize_rows(M: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    n = np.linalg.norm(M, axis=1, keepdims=True)
    return M / (n + eps)

## 6) 贪心 Dartboard 选择（核心算法）

给定：

- `query_distances`：query 到每个候选的距离
- `document_distances`：候选之间两两距离

算法流程：

1. 先选 **最相关**（query 距离最小）的候选
2. 之后每一步选择一个新的候选，使得：
   - 它对 query 仍然相关
   - 同时能带来“新信息”（避免和已选内容太相似）

我们用两个权重控制取舍：

- `RELEVANCE_WEIGHT`：越大越偏“相关”
- `DIVERSITY_WEIGHT`：越大越偏“多样”

In [8]:
# 配置参数
DIVERSITY_WEIGHT = 1.0
RELEVANCE_WEIGHT = 1.0
SIGMA = 0.1


def greedy_dartsearch(
    query_distances: np.ndarray,
    document_distances: np.ndarray,
    documents: List[str],
    num_results: int,
) -> Tuple[List[str], List[float]]:
    """在候选中做贪心选择：相关 + 多样。"""

    sigma = max(SIGMA, 1e-5)

    # 距离 -> 对数分数（越近越大）
    query_scores = lognorm(query_distances, sigma).reshape(-1)  # (n,)
    doc_scores = lognorm(document_distances, sigma)  # (n,n)

    # 先选最相关
    most_relevant_idx = int(np.argmax(query_scores))
    selected = np.array([most_relevant_idx], dtype=int)
    selection_scores: List[float] = [1.0]

    max_dist = doc_scores[most_relevant_idx]  # (n,)

    while len(selected) < num_results:
        # updated_dist: (n,n)  每一行 j 表示“假设选择 j 后”，与其它文档的最大多样性分数
        updated_dist = np.maximum(max_dist, doc_scores)

        combined = updated_dist * DIVERSITY_WEIGHT + query_scores * RELEVANCE_WEIGHT
        # 每个候选 j 得到一个标量分数
        scores = logsumexp(combined, axis=1)
        scores[selected] = -np.inf

        best_idx = int(np.argmax(scores))
        best_score = float(np.max(scores))

        max_dist = updated_dist[best_idx]
        selected = np.append(selected, best_idx)
        selection_scores.append(best_score)

    return [documents[i] for i in selected], selection_scores

## 7) 用 Dartboard 做“多样化检索”

实现方式：

1. 先从向量库取回 `oversampling_factor * k` 个候选（比如 3 倍）
2. 在这批候选中计算：
   - query-candidate 距离
   - candidate-candidate 距离
3. 用 `greedy_dartsearch` 从候选里挑出最终 k 个

注意：我们用 **余弦距离**（cosine distance）：

$$
\text{dist}(a,b)=1-\cos(a,b)=1-\frac{a\cdot b}{\|a\|\,\|b\|}
$$

因此需要先把向量做 L2 归一化。

In [9]:
def get_context_with_dartboard(
    query: str,
    k: int = 5,
    oversampling_factor: int = 3,
) -> Tuple[List[str], List[float]]:
    # 1) 先取回候选
    q_vec = np.array([embeddings.embed_query(query)], dtype=np.float32)
    _, candidate_indices = vectorstore.index.search(q_vec, k=k * oversampling_factor)
    cand_idx = [int(i) for i in candidate_indices[0]]

    candidate_docs = [idx_to_doc(i) for i in cand_idx]
    candidate_texts = [d.page_content for d in candidate_docs]

    # 2) 取回候选向量（优先从 FAISS reconstruct；失败再 fallback 到 embed）
    try:
        cand_vecs = np.array([vectorstore.index.reconstruct(i) for i in cand_idx], dtype=np.float32)
    except Exception:
        cand_vecs = np.array(embeddings.embed_documents(candidate_texts), dtype=np.float32)

    cand_vecs = normalize_rows(cand_vecs)
    q_vec_norm = normalize_rows(q_vec)

    # 3) 距离矩阵
    doc_doc_dist = 1.0 - np.clip(cand_vecs @ cand_vecs.T, -1.0, 1.0)
    q_doc_dist = 1.0 - np.clip((q_vec_norm @ cand_vecs.T).reshape(-1), -1.0, 1.0)

    # 4) Dartboard 选择
    selected_texts, selected_scores = greedy_dartsearch(
        query_distances=q_doc_dist,
        document_distances=doc_doc_dist,
        documents=candidate_texts,
        num_results=k,
    )
    return selected_texts, selected_scores


texts_dart, scores = get_context_with_dartboard(test_query, k=3, oversampling_factor=3)
show_texts(texts_dart)
print("Scores:", scores)

Context 1:
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential 
for life on Earth, as it keeps the planet warm enough to support life. However, human 
activities have intensified this natural process, leading to a warmer climate. 
Fossil Fuels 
Burning fossil fuels for energy releases large amounts of CO2. This includes coal, oil, and 
natural gas used for electricity, heating, and transportation. The industrial revolution marked 
the beginning of a significant increase in fossil fuel consumption, which continues to rise 
today. 
Coal

Context 2:
Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. During the Holocene epoch, 

## 8) （可选）用同一套 QA 提示词对比 top-k vs dartboard

我们把两种方法取回的 chunks 分别拼成上下文，用同一个 QA 提示词回答同一个问题。

In [10]:
qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
qa_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个严谨的问答助手。只能使用给定的上下文回答问题。\n"
            "如果上下文不包含答案，直接回答：我不知道。",
        ),
        ("human", "问题：{question}\n\n上下文：\n{context}\n\n回答："),
    ]
)
qa_chain = qa_prompt | qa_llm | StrOutputParser()


def join_context(texts: List[str]) -> str:
    return "\n\n".join(texts)

question = test_query

ans_topk = qa_chain.invoke({"question": question, "context": join_context(texts_topk)}).strip()
ans_dart = qa_chain.invoke({"question": question, "context": join_context(texts_dart)}).strip()

print("=== Answer with regular top-k context ===")
print(ans_topk)
print("\n=== Answer with dartboard context ===")
print(ans_dart)

=== Answer with regular top-k context ===
The main cause of climate change is the increase in greenhouse gases in the atmosphere, primarily due to human activities that have intensified the natural greenhouse effect.

=== Answer with dartboard context ===
The main cause of climate change is the increase in greenhouse gases in the atmosphere, primarily due to human activities such as burning fossil fuels.
